# Introduction to Natural Language Processing

## Tutorial 4

---

In this tutorial, we will learn how to use PyTorch to create, manipulate and get information from tensors. We will also learn how to build and train a Recurrent Neural Network (RNN) to generate English names.


## Part 1: PyTorch Basics

PyTorch is a machine learning framework based on the Torch library, used for applications such as computer vision and natural language processing, originally developed by Meta AI.


### Importing PyTorch

**Note:** Before running any of the code in this notebook, you should have gone through the [PyTorch setup steps](https://pytorch.org/get-started/locally/).

Let's start by importing PyTorch and checking the version we're using.


In [ ]:
import torch
torch.__version__

'2.8.0+cu126'

### Introduction to tensors

Tensors are the fundamental building block of machine learning. Their job is to represent data in a numerical way.
PyTorch Tensors are just like numpy arrays, but they can run on GPU.

For example, you could represent an image as a tensor with shape `[3, 224, 224]` which would mean `[colour_channels, height, width]`, as in the image has `3` colour channels (red, green, blue), a height of `224` pixels and a width of `224` pixels.


### Creating tensors

There's a whole documentation page dedicated to the [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html) class.

The first thing we're going to create is a **scalar**. A scalar is a single number and in tensor-speak it's a zero dimension tensor.


In [ ]:
# Scalar
scalar = torch.tensor(20)
scalar


tensor(20)

In [ ]:
scalar.ndim


0

A **vector** is a single dimension tensor but can contain many numbers.


In [ ]:
# Vector
vector = torch.tensor([3, 2, 10])
print(f"Vector: {vector}")
print(f"Dimensions: {vector.ndim}")
print(f"Shape: {vector.shape}")


Vector: tensor([ 3,  2, 10])
Dimensions: 1
Shape: torch.Size([3])


A **matrix** has two dimensions:


In [ ]:
# Matrix
MATRIX = torch.tensor([[7, 8, 9],
                       [10, 11, 12]])
print(f"Matrix:\\n{MATRIX}")
print(f"Dimensions: {MATRIX.ndim}")
print(f"Shape: {MATRIX.shape}")


Matrix:\ntensor([[ 7,  8,  9],
        [10, 11, 12]])
Dimensions: 2
Shape: torch.Size([2, 3])


### Random tensors

Let's see how to create a tensor of random numbers using [`torch.rand()`](https://pytorch.org/docs/stable/generated/torch.rand.html).


In [ ]:
# Create a random tensor of size (2, 3, 4)
random_tensor = torch.rand(size=(2, 3, 4))
print(f"Shape: {random_tensor.shape}")
print(f"Dtype: {random_tensor.dtype}")


Shape: torch.Size([2, 3, 4])
Dtype: torch.float32


### Getting information from tensors

Three of the most common attributes you'll want to find out about tensors are:
* `shape` - what shape is the tensor?
* `dtype` - what datatype are the elements within the tensor?
* `device` - what device is the tensor stored on? (usually GPU or CPU)


In [ ]:
# Create a tensor
some_tensor = torch.rand(3, 4)

# Find out details about it
print(some_tensor)
print(f"Shape of tensor: {some_tensor.shape}")
print(f"Datatype of tensor: {some_tensor.dtype}")
print(f"Device tensor is stored on: {some_tensor.device}")


tensor([[0.8157, 0.5542, 0.8572, 0.9848],
        [0.7808, 0.1385, 0.5799, 0.5025],
        [0.4440, 0.0481, 0.4179, 0.1258]])
Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


### Basic operations

Tensors support element-wise operations like addition, subtraction, and multiplication:


In [ ]:
tensor = torch.tensor([1, 2, 3])
print(f"Original: {tensor}")
print(f"Add 10: {tensor + 10}")
print(f"Multiply by 10: {tensor * 10}")
print(f"Element-wise multiply: {tensor * tensor}")


Original: tensor([1, 2, 3])
Add 10: tensor([11, 12, 13])
Multiply by 10: tensor([10, 20, 30])
Element-wise multiply: tensor([1, 4, 9])


### Matrix multiplication

One of the most common operations in deep learning is [matrix multiplication](https://www.mathsisfun.com/algebra/matrix-multiplying.html).

The main two rules for matrix multiplication:
1. The **inner dimensions** must match: `(2, 3) @ (3, 2)` will work
2. The resulting matrix has the shape of the **outer dimensions**: `(2, 3) @ (3, 2)` -> `(2, 2)`


In [ ]:
tensor = torch.tensor([1, 2, 3])

# Element-wise vs Matrix multiplication
print(f"Element-wise: {tensor * tensor}")
print(f"Matrix multiplication: {torch.matmul(tensor, tensor)}")


Element-wise: tensor([1, 4, 9])
Matrix multiplication: 14


### Reshaping tensors

Often you'll want to reshape or change the dimensions of your tensors. Key methods include:

* `torch.reshape(input, shape)` - Reshapes input to shape
* `torch.squeeze(input)` - Removes all dimensions with value 1
* `torch.unsqueeze(input, dim)` - Adds a dimension value of 1 at dim


In [ ]:
# Create a tensor
x = torch.arange(1., 8.)
print(f"Original: {x}, shape: {x.shape}")

# Add an extra dimension
x_reshaped = x.reshape(1, 7)
print(f"Reshaped: {x_reshaped}, shape: {x_reshaped.shape}")


Original: tensor([1., 2., 3., 4., 5., 6., 7.]), shape: torch.Size([7])
Reshaped: tensor([[1., 2., 3., 4., 5., 6., 7.]]), shape: torch.Size([1, 7])


### Moving tensors between devices

Each tensor lives on a particular device: this can be your CPU or one of the GPUs. Tensor computations on GPU are much faster. However, multi-tensor operations require all tensor to be on the same device, so you need to be careful.

In [ ]:
x = x.to('cpu')

In [ ]:
x.device

device(type='cpu')

In [ ]:
x + torch.tensor([0], device="cuda:0")

tensor([1., 2., 3., 4., 5., 6., 7.], device='cuda:0')

---

## Part 2: Building a Generative RNN

Now that we understand PyTorch basics, let's build a Recurrent Neural Network (RNN) that can generate new English names!

### What is an RNN?

A Recurrent Neural Network is a type of neural network designed to work with sequential data. Unlike feedforward neural networks, RNNs have connections that form directed cycles, allowing information to persist.

**Key concept:** At each time step, an RNN takes:
1. An input (e.g., one character)
2. A hidden state from the previous time step

And produces:
1. An output (e.g., prediction for next character)
2. A new hidden state (to be used in the next time step)

This makes RNNs perfect for tasks like text generation, where we want to predict the next character based on all the previous characters we've seen.


### Loading the data

Let's load our list of English names:


In [ ]:
with open("english_names.txt") as f:
    # Remove trailing newline
    names = [line.strip() for line in f]

print(f"Loaded {len(names)} names")
print(f"First 10 names: {names[:10]}")


Loaded 3668 names
First 10 names: ['Abbas', 'Abbey', 'Abbott', 'Abdi', 'Abel', 'Abraham', 'Abrahams', 'Abrams', 'Ackary', 'Ackroyd']


### Character vocabulary

We need to define our character vocabulary - all the possible characters that can appear in names:


In [ ]:
import string

all_letters = string.ascii_letters + " .,;'-"
n_letters = len(all_letters)
print(f"Number of possible characters: {n_letters}")
print(f"Characters: {all_letters}")


Number of possible characters: 58
Characters: abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .,;'-


### Helper functions for converting text to tensors

We need to convert characters to one-hot encoded tensors so our RNN can process them:


In [ ]:
# One-hot matrix of first to last letters (not including EOS) for input
def inputTensor(line):
    tensor = torch.zeros(len(line), n_letters)
    for li in range(len(line)):
        letter = line[li]
        tensor[li][all_letters.find(letter)] = 1
    return tensor

# LongTensor of second letter to end (EOS) for target
def targetTensor(line):
    letter_indexes = [all_letters.find(line[li]) for li in range(1, len(line))]
    letter_indexes.append(n_letters - 1)  # EOS
    return torch.LongTensor(letter_indexes)


Let's test these functions:


In [ ]:
test_name = "David"
print(f"Input tensor shape: {inputTensor(test_name).shape}")
print(f"Input tensor: {inputTensor(test_name)}")
print(f"Target tensor shape: {targetTensor(test_name).shape}")
print(f"Target tensor: {targetTensor(test_name)}")


Input tensor shape: torch.Size([5, 58])
Input tensor: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

## Exercise 1: Understanding the RNN Architecture

Before we implement the RNN, let's understand its structure. Our RNN has the following components:

1. **i2h (input to hidden):** Transforms the combined input and hidden state to a new hidden state
2. **i2o (input to output):** Transforms the combined input and hidden state to an output
3. **o2o (output to output):** Further processes the output
4. **dropout:** Regularization to prevent overfitting
5. **softmax:** Converts outputs to log probabilities

**Question:** Why do we combine the input and hidden state before passing them through linear layers?

Think about what information each contains and why both are important for predicting the next character.


### Your answer here:

(Double-click to edit this cell and write your answer)


## Exercise 2: Implementing the RNN Class

Now let's implement the RNN! You'll need to complete the `__init__` and `forward` methods.

**Task:** Complete the RNN class below by filling in the missing code.

**Hints:**
- In `__init__`: Initialize all the layers (i2h, i2o, o2o, dropout, softmax)
- In `forward`:
  1. Combine input and hidden state using `torch.cat`
  2. Pass through the layers in the right order
  3. Return both output and new hidden state
- Use `nn.Linear` for linear transformations
- Use `nn.Dropout(0.1)` for dropout
- Use `nn.LogSoftmax(dim=0)` for the final activation


In [ ]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size

        # TODO: Initialize the layers
        # self.i2h = nn.Linear(...)
        # self.i2o = nn.Linear(...)
        # self.o2o = nn.Linear(...)
        # self.dropout = nn.Dropout(...)
        # self.softmax = nn.LogSoftmax(...)

        pass  # Remove this line once you implement the method

    def forward(self, input, hidden):
        # TODO: Implement the forward pass
        # 1. Combine input and hidden: input_combined = torch.cat((input, hidden))
        # 2. Pass through i2h to get new hidden state
        # 3. Pass through i2o to get output
        # 4. Combine hidden and output, pass through o2o
        # 5. Apply dropout and softmax
        # 6. Return output and hidden

        pass  # Remove this line once you implement the method

    def initHidden(self):
        return torch.zeros(self.hidden_size)


### Test your RNN

Once you've implemented the RNN, test it with this code:


In [ ]:
# Create RNN
rnn = RNN(n_letters, 128, n_letters)
print(rnn)

# Test with a name
test_name = "David"
input_tensor = inputTensor(test_name)
hidden = rnn.initHidden()

print(f"\\nProcessing name: {test_name}")
for i, char in enumerate(test_name):
    output, hidden = rnn(input_tensor[i], hidden)
    print(f"Character '{char}': output shape = {output.shape}")


RNN()
\nProcessing name: David


TypeError: cannot unpack non-iterable NoneType object

## Exercise 3: Implementing the Training Function

Now let's implement the training function. This function will:
1. Process one name character by character
2. Calculate the loss at each step
3. Backpropagate and update weights

**Task:** Complete the `train` function below.

**Hints:**
- Use `nn.NLLLoss()` as the criterion (Negative Log Likelihood Loss)
- Loop through each character in the input
- Accumulate the loss
- After the loop, call `loss.backward()` to compute gradients
- Manually update parameters using gradient descent: `p.data.add_(p.grad.data, alpha=-learning_rate)`


In [ ]:
import random

# Helper function to get random training examples
def randomTrainingExample():
    line = random.choice(names)
    return inputTensor(line), targetTensor(line)


In [ ]:
criterion = nn.NLLLoss()
learning_rate = 0.0005

def train(input_line_tensor, target_line_tensor):
    # TODO: Implement the training function
    # 1. Unsqueeze the target tensor
    # 2. Initialize hidden state
    # 3. Zero the gradients
    # 4. Initialize loss
    # 5. Loop through each character
    # 6. Backpropagate
    # 7. Update parameters
    # 8. Return
    pass  # Remove this line once you implement the function


### Training the RNN

Once you've implemented the training function, run this cell to train the RNN (it will take a few minutes):


In [ ]:
import time
import math

def timeSince(since):
    now = time.time()
    s = now - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

# Re-initialize the RNN for training
rnn = RNN(n_letters, 128, n_letters)

n_iters = 100000
print_every = 5000
plot_every = 500
all_losses = []
total_loss = 0

start = time.time()

for iter in range(1, n_iters + 1):
    output, loss = train(*randomTrainingExample())
    total_loss += loss

    if iter % print_every == 0:
        print('%s (%d %d%%) %.4f' % (timeSince(start), iter, iter / n_iters * 100, loss))

    if iter % plot_every == 0:
        all_losses.append(total_loss / plot_every)
        total_loss = 0


### Plotting the loss curve

Let's visualize how the loss decreased during training:


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(all_losses)
plt.xlabel('Training iterations (x500)')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.show()


## Exercise 4: Implementing the Generation Function

Now for the exciting part - let's generate new names!

**Task:** Complete the `sample` function that generates a new name starting with a given letter.

**The function should:**
1. Start with the given letter
2. At each step, use the RNN to predict the next character
3. Select the character with the highest probability (using `output.topk(1)`)
4. Stop if we hit the end-of-sequence token or reach max_length
5. Return the generated name

**Hints:**
- Use `torch.no_grad()` context to disable gradient calculation
- Use `output.topk(1)` to get the index of the most probable next character
- Check if `topi == n_letters - 1` to detect end-of-sequence
- Convert the character index back to a letter using `all_letters[topi]`


In [ ]:
max_length = 20

def sample(start_letter='A'):
    # TODO: Implement the sampling function

    pass  # Remove this line once you implement the function


### Generate some names!

Once you've implemented the `sample` function, generate some names:


In [ ]:
print("Generated names:")
for i in range(10):
    print(sample())


### Try different starting letters:


In [ ]:
starting_letters = ['A', 'B', 'C', 'D', 'E']
for letter in starting_letters:
    print(f"Starting with '{letter}': {sample(letter)}")


## Exercise 5: Improving Name Generation (Bonus)

You might notice that the generated names are realistic but somewhat repetitive. This is because we're always choosing the character with the highest probability.

**Task:** Modify the `sample` function to use **sampling with temperature** instead of always picking the top character.

**Temperature sampling:**
- Convert log probabilities to probabilities: `probs = torch.exp(output)`
- Sample from this distribution: `topi = torch.multinomial(probs, 1)[0]`
- Lower temperature (divide output by T > 1) makes it more conservative
- Higher temperature (divide by T < 1) makes it more creative

**Question:** Experiment with different temperature values (0.5, 1.0, 2.0). How does temperature affect the generated names?


In [ ]:
# your code here


## Reflection Questions

1. Why do RNNs process sequences one element at a time rather than all at once?

2. What is the purpose of the hidden state in an RNN?

3. Why do we use log softmax and NLLLoss instead of regular softmax and cross-entropy loss?

4. What could we do to improve the quality of generated names? (Think about architecture changes, training improvements, or data preprocessing)

5. This RNN is "vanilla" - modern NLP uses more advanced architectures like LSTMs, GRUs, and Transformers. What limitations of vanilla RNNs might these architectures address?


### Your answers here:

(Double-click to edit and write your reflections)


## Congratulations!

You've successfully built and trained a character-level RNN for text generation. You've learned:

- PyTorch basics: tensors, operations, and neural network modules
- How RNNs work and process sequential data
- How to implement an RNN from scratch
- How to train a generative model
- How to generate new text using a trained model

This is a foundation for more advanced sequence models used in modern NLP!
